In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

import os
import sys
import subprocess
from pathlib import Path
import tkinter as tk
from tkinter import messagebox, ttk


# ------------------------------------------------------------------
# 1️⃣ مسیرهای پایه
BASE_DIR   = Path('./').resolve().parent          # پوشه‌ی mini-tools
TOOLS_DIR  = BASE_DIR                                   # همان‌جا قرار دارد


# ------------------------------------------------------------------
def discover_tools():
    """
    لیست تمام ابزارها را برمی‌گرداند.
    خروجی: [(نام_ابزار, مسیر_اسکریپت), ...]
    """
    tools = []
    for entry in sorted(TOOLS_DIR.iterdir()):
        if entry.is_dir() and not entry.name.startswith('.'):
            # پیدا کردن اولین فایل .py در پوشه
            py_files = list(entry.glob('*.py'))
            if py_files:
                # اگر بیش از یک فایل باشد، می‌توانید اینجا تغییر دهید
                tools.append((entry.name, py_files[0]))
    return tools


# ------------------------------------------------------------------
class ToolManager(tk.Tk):
    def __init__(self):
        super().__init__()

        self.title("Mini‑Tools Manager")
        self.geometry("450x350")
        self.resizable(False, False)

        # --- عنوان
        tk.Label(self,
                 text="لیست ابزارهای Mini‑Tools",
                 font=("Tahoma", 14),
                 fg="#333").pack(pady=10)

        # --- فهرست (Listbox)
        frame = ttk.Frame(self)
        frame.pack(fill=tk.BOTH, expand=True, padx=15, pady=5)

        self.listbox = tk.Listbox(frame,
                                  height=12,
                                  font=("Consolas", 11))
        self.listbox.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)

        # --- اسکرول
        scrollbar = ttk.Scrollbar(frame,
                                 orient="vertical",
                                 command=self.listbox.yview)
        scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
        self.listbox.configure(yscrollcommand=scrollbar.set)

        # --- اضافه کردن ابزارها به لیست
        self.tools_map = {}   # نگاشت نام→مسیر فایل
        for name, path in discover_tools():
            self.listbox.insert(tk.END, name)
            self.tools_map[name] = path

        # --- دکمه اجرا
        btn_run = ttk.Button(self,
                             text="Run Selected Tool",
                             command=self.run_selected)
        btn_run.pack(pady=10)

        # --- double‑click برای ران کردن سریع
        self.listbox.bind("<Double-1>", lambda e: self.run_selected())

    def run_selected(self):
        """اجرای ابزار انتخاب‌شده"""
        selection = self.listbox.curselection()
        if not selection:
            messagebox.showinfo("هیچ ابزاری انتخاب نشده",
                                "لطفاً یک ابزار را از لیست انتخاب کنید.")
            return

        tool_name   = self.listbox.get(selection)
        script_path = self.tools_map[tool_name]

        # مسیر پوشهٔ ابزار برای cwd
        cwd = script_path.parent

        try:
            # subprocess.Popen() باعث می‌شود که هر ابزار در پروسه‌ی جداگانه اجرا شود.
            subprocess.Popen([sys.executable, str(script_path)],
                             cwd=str(cwd),
                             stdout=subprocess.PIPE,
                             stderr=subprocess.PIPE)

            messagebox.showinfo("در حال اجرا",
                                f"ابزار '{tool_name}' شروع به کار کرده است.")
        except Exception as exc:
            messagebox.showerror("خطا در اجرای ابزار",
                                 f"{exc}\n\nآیا فایل وجود دارد؟")
        

# ------------------------------------------------------------------
if __name__ == "__main__":
    app = ToolManager()
    app.mainloop()
